In [28]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import  pandas as pd 
import json 
import os
from glob import glob
import seaborn as sns 
import numpy as np 
import re
import tikzplotly
import plotly.express as px
from IPython.display import display
from PIL import Image
import matplotlib as mpl
import matplotlib.pyplot as plt 
import plotly 
import plotly.graph_objects as go
# If you want to import all functions from benchmark-utils.py, use the following:
import sys
sys.path.append('../utils')
from benchmark_utils import *

In [29]:


folders = ["../../../zs_out/20251003_004553/",
           "../../../zs_out/no_bucket_20251003_004919/",
           "../../../zs_out/no_bucker_512_20251003_005418/"]

folder=folders[0]


df:pd.DataFrame = merge_benchmark_results(folders)
df = assign_cluster_and_role(df)
df["cpu"] = clean_dool_data(df).Dool.apply(extract_max_cpu_usage)


In [30]:
clients = df.query("Role == 0").groupby("folder")[["Throughput"]].sum().reset_index().sort_values("folder")

fig = go.Figure() 
fig.add_trace(go.Bar(x=clients.folder,y=clients.Throughput.astype(float)))
fig.show()

In [31]:
def plot_max_cpu_by_folder(role):
    folder_cpu_max = df.query("Role == @role ").groupby("folder")[["cpu"]].max().reset_index().sort_values("folder")
    fig = go.Figure() 
    fig.add_trace(go.Bar(x=folder_cpu_max.folder,y=folder_cpu_max.cpu.astype(float)))
    fig.show()

plot_max_cpu_by_folder(2)
plot_max_cpu_by_folder(1)

In [32]:
def generate_pivot_tables_with_bytes(df):
    def generate_bytes_pivot_table(r):
        if r["Role"] == 0:
            return pd.DataFrame()
        df_bytes = pd.DataFrame(r["BytesSentTo"])
        df_bytes["ToRole"] = df_bytes["CID"].apply(lambda cid: r["IDtoRole"][cid])
        pivot = df_bytes.pivot_table(columns='Typ', values='Value', aggfunc='sum')
        pivot['index'] = r.name  # Use row index for merging
        return pivot.reset_index()

# Collect all pivot tables
    pivot_tables = [generate_bytes_pivot_table(row) for _, row in df.iterrows() if row["Role"] != 0]
    pivot_df = pd.concat(pivot_tables, ignore_index=True).fillna(0)

# Merge with original df on index
    df_with_bytes = df.merge(pivot_df, left_index=True, right_on='index', how='left')
    return df_with_bytes

df_with_bytes = generate_pivot_tables_with_bytes(df)


In [33]:
types = ["Checkpoint",	"ClientResponse"		,"Commit"	,"PrePrepare"	,"Prepare"]

average_by_folder_and_role = df_with_bytes.groupby(["folder","Role"]).mean(numeric_only=True).reset_index()
total_bytes_sent = average_by_folder_and_role[["folder","Role"]+types].groupby(["folder","Role"]).sum().sum(axis=1).reset_index().set_index(["folder","Role"])
total_bytes_sent[0].index.set_names(['folder', 'Role'], inplace=True)

normalized_bytes_per_folder = (
    average_by_folder_and_role.set_index(["folder", "Role"])[types]
    .div(total_bytes_sent[0], axis=0)
    .reset_index()
)
normalized_bytes_per_folder["Role"]=average_by_folder_and_role["Role"]
fig = go.Figure()

for typ in types:
    fig.add_trace(go.Bar(x=normalized_bytes_per_folder.Role.apply(str)+normalized_bytes_per_folder.folder.apply(str),y=normalized_bytes_per_folder[typ],name=typ))
fig.update_layout(barmode='stack')
fig.show()

In [34]:
total_bytes_sent[0]/df.groupby(["folder","Role"])[["RunTime"]].mean().RunTime*8*10**(-9)

folder                                          Role
../../../zs_out/20251003_004553/                0       0.000000
                                                1       0.104355
                                                2       1.136804
../../../zs_out/no_bucker_512_20251003_005418/  0       0.000000
                                                1       0.219818
                                                2       0.953682
../../../zs_out/no_bucket_20251003_004919/      0       0.000000
                                                1       0.111282
                                                2       2.313337
dtype: float64

In [35]:
normalized_bytes_per_folder

,folder,Role,Checkpoint,ClientResponse,Commit,PrePrepare,Prepare
0,../../../zs_out/20251003_004553/,0,NaN,NaN,NaN,NaN,NaN
1,../../../zs_out/20251003_004553/,1,0.000969,0.063924,0.448297,0.000000,0.486810
2,../../../zs_out/20251003_004553/,2,0.000088,0.005728,0.040523,0.909676,0.043985
3,../../../zs_out/no_bucker_512_20251003_005418/,0,NaN,NaN,NaN,NaN,NaN
4,../../../zs_out/no_bucker_512_20251003_005418/,1,0.000970,0.063976,0.448357,0.000000,0.486697
5,../../../zs_out/no_bucker_512_20251003_005418/,2,0.000216,0.014042,0.100173,0.776930,0.108638
6,../../../zs_out/no_bucket_20251003_004919/,0,NaN,NaN,NaN,NaN,NaN
7,../../../zs_out/no_bucket_20251003_004919/,1,0.000968,0.063879,0.448214,0.000000,0.486939
8,../../../zs_out/no_bucket_20251003_004919/,2,0.000045,0.002924,0.020888,0.953470,0.022672
